In [ ]:
# =====================================================================
# HEALTHCARE FRAUD DETECTION NOTEBOOK - MILESTONE 2
# =====================================================================
# Basic imports
import os
import requests

In [ ]:
# =====================================================================
# 1. REQUIREMENTS, IMPORTS & DATA RETRIEVAL
# =====================================================================
## 1. Clone the repository
REPO_URL = "https://github.com/Jheff-AIML/data-and-infrastructure-2026_project_m2.git"
REPO_NAME = "data-and-infrastructure-2026_project_m2"

if not os.path.exists(REPO_NAME):
    !git clone {REPO_URL}

## 2. Change the working directory to the cloned repository
%cd {REPO_NAME}

## 3. Install the requirements
if os.path.exists("requirements.txt"):
    !pip install -r requirements.txt
else:
    print("requirements.txt not found!")

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import klib
from google.cloud import storage
from google.cloud import bigquery
from sklearn.model_selection import train_test_split, cross_validate, StratifiedKFold, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, TargetEncoder
from sklearn.metrics import precision_recall_curve, classification_report, roc_auc_score
import xgboost as xgb

In [ ]:
# --- GCP Configuration ---
PROJECT_ID = "dh-inf-for-ai-509122"    # Set the GCS project id
BUCKET_NAME = "milestone_two"  # Set the GCS bucket name
LOCATION = "EU"                        # Set the target data region/jurisdiction

# --- Authentication options ---
# Option A: Google Colab / Vertex AI:
from google.colab import auth
auth.authenticate_user()

print(f"Target project: {PROJECT_ID}")
print(f"Target bucket: gs://{BUCKET_NAME}")

In [ ]:
def create_gcs_bucket(bucket_name, project_id, location="EU"):
    """Create a GCS bucket if it does not already exist with production defaults."""
    client = storage.Client(project=project_id)

    # Return gracefully if bucket is already present
    if client.lookup_bucket(bucket_name):
        print(f"Bucket 'gs://{bucket_name}' already exists.")
        return

    # 1. Initialize the local bucket structure contract
    bucket = client.bucket(bucket_name)
    bucket.iam_configuration.uniform_bucket_level_access_enabled = True
    bucket.versioning_enabled = True

    # 2. FIX: Use the native helper method to register object lifecycle actions safely
    # age is defined explicitly in days, is_live controls noncurrent version archiving
    bucket.add_lifecycle_delete_rule(
        age=7,
        is_live=False
    )

    # 3. Commit deployment execution directly to your GCP project environment
    bucket = client.create_bucket(bucket, location=location)
    print(f"Successfully created bucket: gs://{bucket.name}")


create_gcs_bucket(BUCKET_NAME, PROJECT_ID, LOCATION)

In [ ]:
import os
from datetime import date
import kagglehub
from google.colab import userdata
userdata.get('kaggle')
# Download the dataset folder
dataset_folder = kagglehub.dataset_download("nudratabbas/healthcare-fraud-detection-dataset")
print("Dataset folder path:", dataset_folder)
files = os.listdir(dataset_folder)
print("Files in folder:", files)
# Robust CSV pick: first *.csv FILE (not a subfolder) — safer than files[0]
csv_filename = next(f for f in files
                    if f.endswith(".csv") and os.path.isfile(os.path.join(dataset_folder, f)))
DATASET_LOCAL_RAW_PATH = os.path.join(dataset_folder, csv_filename)
print("Full local path to file:", DATASET_LOCAL_RAW_PATH)
RETRIEVAL_DATE = date.today().isoformat()
# UPGRADE 2 (provenance manifest): hash + shape, uploaded beside the raw object.
import hashlib, json   # pandas already imported in the imports cell
with open(DATASET_LOCAL_RAW_PATH, "rb") as f:
    sha256 = hashlib.sha256(f.read()).hexdigest()
df_check = pd.read_csv(DATASET_LOCAL_RAW_PATH)
print("sha256:", sha256)
print("shape:", df_check.shape)
manifest = {
    "dataset": "nudratabbas/healthcare-fraud-detection-dataset",
    "kagglehub_version": 1,      # UPGRADE 1: fill with the pinned version int
    "retrieval_date": RETRIEVAL_DATE,
    "source_file": csv_filename,
    "sha256": sha256,
    "rows": int(df_check.shape[0]),
    "cols": int(df_check.shape[1]),
    "file_bytes": os.path.getsize(DATASET_LOCAL_RAW_PATH),
}
print(json.dumps(manifest, indent=2))
def upload_to_gcs(bucket_name, source_file_name, destination_blob_name):
    client = storage.Client()
    blob = client.bucket(bucket_name).blob(destination_blob_name)
    blob.upload_from_filename(source_file_name)
    print(f"Uploaded {source_file_name} -> gs://{bucket_name}/{destination_blob_name}")
# Upload the raw object AND its manifest into bronze
upload_to_gcs(BUCKET_NAME, DATASET_LOCAL_RAW_PATH, f"{RETRIEVAL_DATE}/bronze/healthcare_fraud.csv")
with open("manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)
upload_to_gcs(BUCKET_NAME, "manifest.json", f"{RETRIEVAL_DATE}/bronze/manifest.json")
os.remove("manifest.json")  # tidy the runtime; the bronze copy is the record

Bronze stage complete



In [ ]:
# =====================================================================
# SILVER ENTRY — read BRONZE from GCS (NOT the local source), verify
# against the manifest, and ASSERT the schema contract (validated, not blind)
# =====================================================================
import hashlib, json, io
# storage.Client() and BUCKET_NAME carry over from earlier cells

BRONZE_DATE = RETRIEVAL_DATE      # the date bronze was uploaded (today)
BRONZE_DIR  = f"{BRONZE_DATE}/bronze"
client = storage.Client()

def read_gcs_bytes(bucket, path):
    return client.bucket(bucket).blob(path).download_as_bytes()

# --- pull the raw object AND its manifest from GCS bronze ---
csv_bytes  = read_gcs_bytes(BUCKET_NAME, f"{BRONZE_DIR}/healthcare_fraud.csv")
manifest   = json.loads(read_gcs_bytes(BUCKET_NAME, f"{BRONZE_DIR}/manifest.json"))

# --- INTEGRITY GATE: what we pulled == what we stored ---
pulled_sha = hashlib.sha256(csv_bytes).hexdigest()
assert pulled_sha == manifest["sha256"], "bronze integrity check FAILED (hash mismatch)"
df = pd.read_csv(io.BytesIO(csv_bytes))
assert df.shape == (manifest["rows"], manifest["cols"]), "shape mismatch vs manifest"
print("integrity ok: pulled sha256 == manifest; shape", df.shape)

# --- SCHEMA CONTRACT: ASSERT (earned by EDA), don't blindly impose ---
EXPECTED_COLS = ["Provider_ID","Claim_ID","Patient_Age","Patient_Gender",
                 "Diagnosis_Code","Procedure_Code","Claim_Amount","Approved_Amount",
                 "Insurance_Type","Claim_Submission_Date","Days_Between_Service_and_Claim",
                 "Number_of_Claims_Per_Provider_Monthly","Provider_Specialty",
                 "Patient_State","Claim_Status","Is_Fraud","Length_of_Stay",
                 "Visit_Type","Chronic_Condition_Flag","Prior_Visits_12m"]
assert list(df.columns) == EXPECTED_COLS, f"column drift: {set(EXPECTED_COLS) ^ set(df.columns)}"
print("schema contract ok:", len(EXPECTED_COLS), "columns, exact match")

# --- WAVE 1 CASTING (validated by EDA) ---
df["Is_Fraud"] = df["Is_Fraud"].astype(int)            # target: binary 0/1
explicit_float_cols = ["Claim_Amount", "Approved_Amount", "Length_of_Stay"]
df[explicit_float_cols] = df[explicit_float_cols].astype(float)

print("silver entry done → next waves: validate -> impute -> engineer")
df.head()

In [ ]:
# =====================================================================
# SILVER WAVE 2 — VALIDATE + IMPUTE the ~350 nulls (decisions D1, D2)
# Justification (EV1/EV7): missingness is CLASS-FLAT — missing->fraud rate
# for Prior_Visits 8.3% / Insurance 7.1% / Specialty 8.6% vs baseline 8.3%,
# so imputing introduces no class skew. Co-missing cells are tiny (<=13), so
# we deliberately do NOT engineer missing-indicator features (EV1/EV7).
# =====================================================================
NULL_COLS = ["Prior_Visits_12m", "Insurance_Type", "Provider_Specialty"]
IMPUTE_MAP = {
    "Prior_Visits_12m": 0,              # D1: count feature; null ~= no prior visits
    "Insurance_Type": "Unknown",        # D2: categorical token
    "Provider_Specialty": "Unknown",    # D2: categorical token
}
orig_shape = df.shape

# --- validate BEFORE: class-conditional missing vs present must be flat ---
print("nulls before:", df[NULL_COLS].isnull().sum().to_dict())
for c in NULL_COLS:
    m = df[c].isnull()
    print(f"  {c:20s} missing->fraud={df.loc[m,'Is_Fraud'].mean():.2%}  "
          f"present->fraud={df.loc[~m,'Is_Fraud'].mean():.2%}  (baseline {df['Is_Fraud'].mean():.2%})")

# --- impute (per D1/D2) ---
for c, fill in IMPUTE_MAP.items():
    df[c] = df[c].fillna(fill)

# --- validate AFTER ---
assert df[NULL_COLS].isnull().sum().sum() == 0, "imputation left nulls behind"
assert df.shape == orig_shape, "row count changed during imputation"
assert df["Is_Fraud"].dtype == int, "target type should be int (set in entry cell)"
print("nulls after impute:", df[NULL_COLS].isnull().sum().sum())
print("shape preserved:", df.shape)
print("impute wave done -> next: close-of-silver write, then gold")

In [ ]:
# =====================================================================
# SILVER CLOSE — materialize the cleaned silver tier to GCS (D17, Option A)
# Input : df  (the silver frame, after cast + validate + impute waves)
# Output: gs://{BUCKET_NAME}/{date}/silver/healthcare_fraud_clean.csv + manifest.json
# Bronze is NOT touched here — it stays raw and immutable.
# =====================================================================
import hashlib, json
SILVER_DATE = RETRIEVAL_DATE            # this run, same as bronze ingestion
BRONZE_DATE = RETRIEVAL_DATE            # the bronze this silver was derived from
SILVER_DIR  = f"{SILVER_DATE}/silver"
# --- pre-write validation: silver must be final and complete ---
assert df[["Prior_Visits_12m", "Insurance_Type", "Provider_Specialty"]].isnull().sum().sum() == 0, "nulls remain"
assert len(df) == 10000, "silver row count must equal bronze (10000)"
print("silver frame ok:", df.shape, "nulls=0")
# --- serialize to bytes so we hash the exact object we upload ---
csv_bytes = df.to_csv(index=False).encode("utf-8")
sha256 = hashlib.sha256(csv_bytes).hexdigest()
# --- silver manifest: provenance chained back to the bronze it came from ---
bronze_manifest = json.loads(read_gcs_bytes(BUCKET_NAME, f"{BRONZE_DATE}/bronze/manifest.json"))
silver_manifest = {
    "layer": "silver",
    "dataset": "nudratabbas/healthcare-fraud-detection-dataset",
    "source_bronze": f"gs://{BUCKET_NAME}/{BRONZE_DATE}/bronze/healthcare_fraud.csv",
    "source_bronze_sha256": bronze_manifest["sha256"],     # links silver to its raw parent
    "retrieval_date": SILVER_DATE,
    "transformations_applied": [
        "cast: Is_Fraud->int; Claim_Amount/Approved_Amount/Length_of_Stay->float",
        "impute: Prior_Visits_12m->0; Insurance_Type/Provider_Specialty->'Unknown' (D1/D2)",
        "validate: integrity gate vs bronze manifest; schema contract; 0 nulls; rows preserved"
    ],
    "sha256": sha256,
    "rows": df.shape[0],
    "cols": df.shape[1],
    "file_bytes": len(csv_bytes),
}
print(json.dumps(silver_manifest, indent=2))
# --- upload the clean object AND its manifest into silver ---
with open("healthcare_fraud_clean.csv", "wb") as f:
    f.write(csv_bytes)                       # exact bytes we hashed
upload_to_gcs(BUCKET_NAME, "healthcare_fraud_clean.csv", f"{SILVER_DIR}/healthcare_fraud_clean.csv")
with open("silver_manifest.json", "w") as f:
    json.dump(silver_manifest, f, indent=2)
upload_to_gcs(BUCKET_NAME, "silver_manifest.json", f"{SILVER_DIR}/manifest.json")
os.remove("healthcare_fraud_clean.csv"); os.remove("silver_manifest.json")
print("silver tier materialized -> gs://%s/%s/" % (BUCKET_NAME, SILVER_DIR))

In [ ]:
# =====================================================================
#  GOLD TIER — model-ready feature matrix (reads SILVER, not bronze)
#  Mirror of the silver-entry GCS idiom; no invented helpers.
# =====================================================================
import hashlib, json, io
# storage.Client(), BUCKET_NAME, read_gcs_bytes carry over from the silver cell
from datetime import date

SILVER_DATE = RETRIEVAL_DATE   # the {date} silver was written under (adjust if different)

# --- pull the CLEANED silver object + its manifest from GCS silver ---
s_dir = f"{SILVER_DATE}/silver"
csv_bytes   = read_gcs_bytes(BUCKET_NAME, f"{s_dir}/healthcare_fraud_clean.csv")
silver_meta = json.loads(read_gcs_bytes(BUCKET_NAME, f"{s_dir}/manifest.json"))

gold = pd.read_csv(io.BytesIO(csv_bytes))

# --- INTEGRITY GATE vs the SILVER manifest (silver already chains to bronze) ---
assert hashlib.sha256(csv_bytes).hexdigest() == silver_meta["sha256"], "silver integrity FAILED"
assert gold.shape == (silver_meta["rows"], silver_meta["cols"]), "shape mismatch vs silver manifest"
print("silver integrity ok — passed sha256 + shape; gold input is clean & imputed")

# --- (1) time features (D3); drop the raw date ---
sub = pd.to_datetime(gold["Claim_Submission_Date"])
gold["Submission_Month"] = sub.dt.month.astype(int)
gold["Is_Submission_Weekend"] = sub.dt.weekday.isin([5, 6]).astype(int)
gold = gold.drop(columns=["Claim_Submission_Date"])

# --- (2) stratified split 80/10/10 + tag (D6/D15) ---
train, temp = train_test_split(gold, test_size=0.20, random_state=42, stratify=gold["Is_Fraud"])
dev,  test  = train_test_split(temp,  test_size=0.50, random_state=42, stratify=temp["Is_Fraud"])
train["split"], dev["split"], test["split"] = "train", "dev", "test"

# --- (3) velocity profiling on TRAIN ONLY (D7/D14; 'fast' = <=6 days) ---
stats = (train.groupby("Provider_ID")
         .agg(Hist_Pct_Fast_Claims=("Days_Between_Service_and_Claim", lambda x: (x <= 6).mean()),
              Hist_Mean_Lag=("Days_Between_Service_and_Claim", "mean"),
              Hist_Lag_Std=("Days_Between_Service_and_Claim", "std"))
         .reset_index())
stats["Hist_Lag_Std"] = stats["Hist_Lag_Std"].fillna(0.0)          # providers seen once

g_fast = (train["Days_Between_Service_and_Claim"] <= 6).mean()      # unseen-provider fallbacks
g_mean = train["Days_Between_Service_and_Claim"].mean()
g_std  = train["Days_Between_Service_and_Claim"].std()

for name, df_ in (("train", train), ("dev", dev), ("test", test)):
    df_ = df_.merge(stats, on="Provider_ID", how="left")
    df_["Hist_Pct_Fast_Claims"]    = df_["Hist_Pct_Fast_Claims"].fillna(g_fast)
    df_["Hist_Mean_Lag"]           = df_["Hist_Mean_Lag"].fillna(g_mean)
    df_["Hist_Lag_Std"]            = df_["Hist_Lag_Std"].fillna(g_std)
    df_["Current_vs_Hist_Mean_Diff"] = df_["Days_Between_Service_and_Claim"] - df_["Hist_Mean_Lag"]
    df_["Current_Speed_Z_Score"]     = df_["Current_vs_Hist_Mean_Diff"] / (df_["Hist_Lag_Std"] + 1e-5)
    # ASSIGN BACK by name — merge() returns a copy; the loop var alone would lose it
    if name == "train":   train = df_
    elif name == "dev":   dev   = df_
    else:                 test  = df_

gold_full = pd.concat([train, dev, test], ignore_index=True)

# --- (4) deterministic amount transform at GOLD (D13); scaling fits per-fold in pipeline ---
for c in ("Claim_Amount", "Approved_Amount"):
    gold_full[c] = np.log1p(gold_full[c])

# --- (5) assemble pre-encoding feature matrix + target + split (D16) ---
FEATURES = [
    # categorical (encoded fit-per-fold in the training pipeline)
    "Diagnosis_Code", "Procedure_Code", "Insurance_Type", "Visit_Type",
    "Provider_Specialty", "Patient_State",
    # numeric (amounts already log1p'd here; scaler stays in pipeline)
    "Claim_Amount", "Approved_Amount", "Number_of_Claims_Per_Provider_Monthly",
    "Length_of_Stay", "Chronic_Condition_Flag", "Prior_Visits_12m",
    "Hist_Pct_Fast_Claims", "Hist_Mean_Lag", "Hist_Lag_Std",
    "Current_vs_Hist_Mean_Diff", "Current_Speed_Z_Score",
    "Submission_Month", "Is_Submission_Weekend",
]
gold_matrix = gold_full[FEATURES + ["Is_Fraud", "split"]]
assert gold_matrix.isnull().sum().sum() == 0
assert len(gold_matrix) == 10000 and list(gold_matrix["split"].value_counts().values) == [8000, 1000, 1000]
assert set(gold_matrix["split"].unique()) == {"train", "dev", "test"}

# --- write gold + manifest (inline via storage client — matches your idiom) ---
def _write_bytes_to_gcs(data, local_name, dest_path):
    with open(local_name, "w", encoding="utf-8") as f:
        f.write(data)
    client.bucket(BUCKET_NAME).blob(dest_path).upload_from_string(data)
    print("uploaded", dest_path)

GOLD_DATE = date.today().isoformat()
GOLD_DIR  = f"{GOLD_DATE}/gold"

csv_str = gold_matrix.to_csv(index=False)
_gold_csv_sha = hashlib.sha256(csv_str.encode("utf-8")).hexdigest()

gold_manifest = {
    "rows": len(gold_matrix), "cols": len(gold_matrix.columns),
    "sha256": _gold_csv_sha,
    "features": FEATURES, "target": "Is_Fraud", "split": "split",
    "split_counts": {k: int(v) for k, v in gold_matrix["split"].value_counts().items()},
    "fraud_rate": float(gold_matrix["Is_Fraud"].mean()),
    "source_silver": f"gs://{BUCKET_NAME}/{SILVER_DATE}/silver/healthcare_fraud_clean.csv",
}

_write_bytes_to_gcs(csv_str, "healthcare_fraud_gold.csv", f"{GOLD_DIR}/healthcare_fraud_gold.csv")
gold_matrix.to_parquet("healthcare_fraud_gold.parquet", index=False)
client.bucket(BUCKET_NAME).blob(f"{GOLD_DIR}/healthcare_fraud_gold.parquet")\
      .upload_from_filename("healthcare_fraud_gold.parquet")
print("uploaded", f"{GOLD_DIR}/healthcare_fraud_gold.parquet")
_write_bytes_to_gcs(json.dumps(gold_manifest, indent=2), "manifest.json", f"{GOLD_DIR}/manifest.json")

print("GOLD CLOSED —", gold_matrix.shape, "| fraud", round(gold_matrix["Is_Fraud"].mean(), 4),
      "| splits", dict(gold_matrix["split"].value_counts()))

silver integrity ok — passed sha256 + shape; gold input is clean & imputed
uploaded 2026-11-02/gold/healthcare_fraud_gold.csv
uploaded 2026-11-02/gold/healthcare_fraud_gold.parquet
uploaded 2026-11-02/gold/manifest.json
GOLD CLOSED — (10000, 21) | fraud 0.0829 | splits {'train': np.int64(8000), 'dev': np.int64(1000), 'test': np.int64(1000)}
